p# PU-Bagging reliable non-landslide selection

This notebook first cleans raster NoData gaps inside the study area, then uses the cleaned rasters for PU-Bagging. The PU-Bagging workflow still saves only one final CSV: `data/processed/pu_bagging/reliable_nonlandslide_points_all.csv`.

## 1. Load input paths and parameters

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pu_bagging import (
    PUBaggingConfig,
    attach_valid_factor_values,
    generate_valid_unlabeled_candidates,
    load_landslide_points,
    read_raster_factors,
    read_study_boundary,
    run_pu_bagging,
    save_reliable_nonlandslides,
    select_reliable_nonlandslides,
)
from src.raster_cleaning import (
    COMMON_NODATA,
    audit_cleaned_rasters,
    clean_all_rasters,
)

raw_raster_dir = PROJECT_ROOT / "data/raw/rasters"
cleaned_raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
boundary_dir = PROJECT_ROOT / "data/raw/boundary"
samples_dir = PROJECT_ROOT / "data/raw/samples"

landslide_points_csv = samples_dir / "landslide_points.csv"
if not landslide_points_csv.exists():
    fallback_csvs = sorted(samples_dir.glob("landslide*.csv"))
    if len(fallback_csvs) == 1:
        landslide_points_csv = fallback_csvs[0]
        print(f"Using landslide point CSV: {landslide_points_csv}")

config = PUBaggingConfig(
    project_root=PROJECT_ROOT,
    raster_dir=cleaned_raster_dir,
    landslide_points_csv=landslide_points_csv,
    samples_dir=samples_dir,
    boundary_dir=boundary_dir,
    output_csv=PROJECT_ROOT / "data/processed/pu_bagging/reliable_nonlandslide_points_all.csv",
    n_unlabeled_candidates=50_000,
    n_iterations=100,
    temp_negative_ratio=1.0,
    tau=0.5,
    reliable_vote_ratio_threshold=0.5,
    landslide_point_buffer_m=90.0,
    random_seed=42,
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    max_attempts=1_000_000,
).resolve()

print(f"Project root: {config.project_root}")
print(f"Raw raster folder: {raw_raster_dir}")
print(f"Cleaned raster folder: {config.raster_dir}")
print(f"Final PU-Bagging CSV: {config.output_csv}")

## 2. Raster NoData cleaning and validation

In [ ]:
cleaning_summary = clean_all_rasters(
    raster_dir=raw_raster_dir,
    boundary_dir=boundary_dir,
    cleaned_dir=cleaned_raster_dir,
    common_nodata=COMMON_NODATA,
)

cleaned_audit = audit_cleaned_rasters(cleaned_raster_dir, common_nodata=COMMON_NODATA)
display_columns = [
    "raster",
    "detected_type",
    "original_nodata",
    "nodata_inside_before",
    "nodata_inside_percent_before",
    "nodata_inside_after",
    "output_path",
    "status",
]
print(cleaning_summary[display_columns].to_string(index=False))
print(f"Cleaned raster audit passed: {cleaned_audit['passes']}")
if cleaned_audit["errors"]:
    print("Audit errors:")
    for error in cleaned_audit["errors"]:
        print(f"- {error}")

## 3. Read and check cleaned raster factors

In [ ]:
raster_stack = read_raster_factors(config.raster_dir)
factor_names = list(raster_stack.factor_names)

print(f"Cleaned raster factors found: {len(raster_stack.paths)}")
for factor_name, path in zip(factor_names, raster_stack.paths):
    print(f"{factor_name}: {path.name}")
print(f"CRS: {raster_stack.crs}")
print(f"Resolution: {raster_stack.resolution}")
print(f"NoData: {raster_stack.nodata}")

## 4. Load landslide points

In [ ]:
landslide_points = load_landslide_points(config.landslide_points_csv, raster_stack.crs)
valid_landslide_points = attach_valid_factor_values(landslide_points, raster_stack)

print(f"Loaded landslide points: {len(landslide_points)}")
print(f"Valid landslide points after raster extraction: {len(valid_landslide_points)}")

## 5. Load study area boundary

In [ ]:
boundary = read_study_boundary(config.boundary_dir, raster_stack.crs)

print(f"Boundary features: {len(boundary)}")
print(f"Boundary CRS: {boundary.crs}")

## 6. Generate valid unlabeled candidate points

In [ ]:
print(f"Target valid unlabeled candidates: {config.n_unlabeled_candidates}")
print(f"Maximum random point attempts: {config.max_attempts}")
print("Candidates will be sampled inside the boundary and outside landslide exclusions.")

## 7. Extract factor values

In [ ]:
unlabeled_candidates = generate_valid_unlabeled_candidates(
    boundary=boundary,
    landslide_points=valid_landslide_points,
    raster_stack=raster_stack,
    config=config,
)

print(f"Valid unlabeled candidates: {len(unlabeled_candidates)}")

## 8. Run PU-Bagging

In [ ]:
pu_result = run_pu_bagging(
    positive_points=valid_landslide_points,
    unlabeled_candidates=unlabeled_candidates,
    factor_names=factor_names,
    config=config,
)

print(f"PU-Bagging iterations completed: {config.n_iterations}")

## 9. Select reliable non-landslide points

In [ ]:
reliable_nonlandslides = select_reliable_nonlandslides(
    unlabeled_candidates=unlabeled_candidates,
    pu_result=pu_result,
    factor_names=factor_names,
    reliable_vote_ratio_threshold=config.reliable_vote_ratio_threshold,
)

reliable_percentage = 100 * len(reliable_nonlandslides) / len(unlabeled_candidates)
print(f"Reliable non-landslide points selected: {len(reliable_nonlandslides)}")
print(f"Reliable non-landslide percentage: {reliable_percentage:.2f}%")

## 10. Save the single final CSV

In [ ]:
saved_csv = save_reliable_nonlandslides(
    reliable_nonlandslides=reliable_nonlandslides,
    output_csv=config.output_csv,
    factor_names=factor_names,
)

print("PU-Bagging reliable non-landslide selection summary")
print(f"number of valid landslide points: {len(valid_landslide_points)}")
print(f"number of valid unlabeled candidates: {len(unlabeled_candidates)}")
print(f"number of PU-Bagging iterations: {config.n_iterations}")
print(f"number of reliable non-landslide points selected: {len(reliable_nonlandslides)}")
print(f"percentage of reliable non-landslide points: {reliable_percentage:.2f}%")
print(f"path of the saved CSV: {saved_csv}")